In [6]:
import gzip
import networkx as nx
import json
from networkx.readwrite import json_graph

In [7]:
def parse_rfa(file_path):
    with open(file_path, 'rt', encoding='utf-8') as f:
        content = f.read()

    entries = content.strip().split("\n\n")
    data = []

    for entry in entries:
        record = {}
        for line in entry.split("\n"):
            if ":" in line:
                key, value = line.split(":", 1)
                record[key.strip()] = value.strip()
        data.append(record)

    return data

In [8]:
def build_graph(data):
    G = nx.DiGraph()

    for d in data:
        src = d["SRC"]
        tgt = d["TGT"]

        G.add_edge(
            src,
            tgt,
            vote=int(d["VOT"]),
            text=d["TXT"],
            result=int(d["RES"]),
            year=int(d["YEA"]),
            date=d["DAT"]
        )

    return G

In [9]:
data = parse_rfa("wiki-RfA.txt")
G = build_graph(data)

print(G.number_of_nodes(), G.number_of_edges())

11381 189003


In [10]:
data = json_graph.node_link_data(G)

with open("rfa_graph.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

In [11]:
len(data) == G.number_of_edges()

False

In [12]:
vote_breakdown = (
    df.groupby(["TGT", "VOT"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

vote_breakdown = vote_breakdown.rename(columns={
    -1: "oppose_votes",
     0: "neutral_votes",
     1: "support_votes"
})

vote_breakdown["total_votes_received"] = (
    vote_breakdown.get("oppose_votes", 0)
    + vote_breakdown.get("neutral_votes", 0)
    + vote_breakdown.get("support_votes", 0)
)

vote_breakdown = vote_breakdown.sort_values("total_votes_received", ascending=False)

print(vote_breakdown.head(20))

NameError: name 'df' is not defined